# STARRT: third GPU round

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. *Add Input* → the `starrt-gpu3` dataset.

Run cell 1 and the smoke test first. Then **Save Version → Save & Run All (Commit)**. Total about 6 hours. Results → `starrt_gpu3_results.zip`.

In [ ]:
import glob, os, subprocess
hits = glob.glob('/kaggle/input/**/run_distilbert.py', recursive=True)
assert hits, 'Dataset not attached: Add Input -> starrt-gpu3'
CODE = os.path.dirname(hits[0]); print('code folder:', CODE)
import torch; print('GPUs:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

def run(script, args=''):
    cmd = f'cd /kaggle/working && python -u {CODE}/{script} {args}'
    print('>>', cmd, flush=True); r = subprocess.run(cmd, shell=True)
    if r.returncode != 0: print('!! FAILED:', script, args, '(continuing with the next step)')

## 0. Smoke test (3 min)

In [ ]:
run('run_distilbert.py', '--model jjzha/jobbert-base-cased --datasets faruque --folds 0 --epochs 1 --limit 200 --skip_forms --seed 13 --tag smoke')
run('run_llm_zeroshot.py', '--datasets faruque --folds 1 --limit 2 --tag smoke_llm')
print('smoke test done: check above that both printed a [smoke ...] result line')

## A. JobBERT with two more seeds (13 and 7), Djinni and Faruque (~2.5 h)

In [ ]:
for seed in [13, 7]:
    run('run_distilbert.py', f'--model jjzha/jobbert-base-cased --datasets djinni,faruque --seed {seed} --tag jobbert_seed{seed} --skip_forms')

## B. Qwen2.5-3B (default prompt) on samples of folds 1-4 (~1 h)
Fold 0 was run before; together this covers all five folds.

In [ ]:
run('run_llm_zeroshot.py', '--datasets djinni,faruque --folds 1,2,3,4 --tag llm_qwen2.5-3b_zeroshot')

## C. Qwen2.5-7B (default prompt, both GPUs) on samples of folds 1-4 (~2 h)

In [ ]:
run('run_llm_zeroshot.py', '--model Qwen/Qwen2.5-7B-Instruct --device_map auto --datasets djinni,faruque --folds 1,2,3,4 --tag llm_qwen2.5-7b_zeroshot')

## Zip results
Download `starrt_gpu3_results.zip` from the **Output** tab and send it in the chat.

In [ ]:
import shutil
for f in glob.glob('/kaggle/working/results/smoke*'): os.remove(f)
shutil.make_archive('/kaggle/working/starrt_gpu3_results', 'zip', '/kaggle/working/results')
print(sorted(os.listdir('/kaggle/working/results')))